In [4]:
from tradejournal.models.factory import create_repository
from os import environ

REPOSITORY_NAME = environ.get('REPOSITORY_NAME', 'azurestorage')
REPOSITORY_SETTINGS = {
        'STORAGE_NAME': environ.get('STORAGE_NAME', ''),
        'CONNECTION_STRING' : environ.get('CONNECTION_STRING', '')
}

repository = create_repository(REPOSITORY_NAME, REPOSITORY_SETTINGS)

In [5]:
journalentries=repository.get_journalentries_forview()
journalentrygroups=repository.get_journalentrygroups_forview(journalentries)
journalentries.extend(journalentrygroups)
journalentries.sort(key = lambda x: (x.is_open(), x.entry_time), reverse=True)

In [59]:
for jg in journalentrygroups:
    print(jg.name)

Ice-cream Bucket Nov24
Manual Jan25
KumoBreakout Jan25
Ice-cream bucket Jan25
Covered Call Jan25
Iron Condor Jan25
Blacksmith Jan25
MoneyInBank Jan25
Covered Call Jan25
KumoBreakout Dec24
Iron Condor Dec24
Blacksmith Dec24
Manual Dec24
Ice-cream Bucket Dec24
MoneyInBank Dec24
CoveredCall Dec24
KumoBreakout Nov24
Iron Condor Nov24 EICHERMOT
Blacksmith Oct24 Set 2
Blacksmith Nov24
MoneyInBank Nov24
Iron Condor Nov24
Manual Nov24
CoveredCall Nov24
IronCondor Nov24 SBIN
Ice-cream Bucket FY2025
Ice-cream Bucket Oct24
MoneyInTheBank Oct24
KumoBreakout Oct24
Manual Oct24
IronCondor Oct24
Stopped - Ratio Spread FY2024
Covered Call Oct24
Blacksmith Oct24
Ice-cream Bucket Sep24
Ratio Spread Sep24
IronCondor Sep24
Iron Condor SBIN Sep24
Blacksmith Sep24
KumoBreakout Sep24
Iron Condor FY2025
Manual Sep24
MoneyInBank Sep24
CoveredCall Sep24
IronCondor NIFTY Sep24
KumoBreakout Aug24
MoneyInTheBank Aug24 Set 2
IronCondor Aug24
NiftyBuy FY2025
Blacksmith Aug24
Manual Aug24
HeikenAshi Aug24
Covered Cal

In [61]:
def fixup_strategy(name):
    name = name.lower()
    if name.startswith('ironcondor') or name.startswith('iron condor'):
        return 'iron condor'
    if name.startswith('moneyinbank') or name.startswith('moneyinthebank'):
        return 'money in the bank'
    if name.startswith('coveredcall') or name.startswith('covered call'):
        return 'covered call'
    if name.startswith('blacksmith'):
        return 'blacksmith'
    if name.startswith('kumobreakout'):
        return 'kumo breakout'
    else:
        return name
    
def recurse_group(g, output):
    for item in g.deserialized_items:
        if item.is_group():
            recurse_group(item, output)
        else:
            output[item.tradingsymbol] = fixup_strategy(g.name.rsplit(' ', 1)[0].lower())

output = {}
for group in journalentrygroups:
    recurse_group(group, output)
output

{'MIDCPNIFTY25JAN12300PE': 'manual',
 'NIFTY2510923800CE': 'manual',
 'NIFTY2510924000PE': 'manual',
 'PETRONET25JAN340PE': 'manual',
 'PETRONET25JANFUT': 'manual',
 'MIDCPNIFTY25JAN12900PE': 'manual',
 'ICICIBANK25JAN1310PE': 'manual',
 'PETRONET25JAN335PE': 'manual',
 'HINDPETRO25JAN420CE': 'manual',
 'HINDPETRO25JAN410CE': 'manual',
 'SYNGENE25JAN860PE': 'kumo breakout',
 'SYNGENE25JANFUT': 'kumo breakout',
 'NIFTY2510923800PE': 'ice-cream bucket',
 'NIFTY2510924100PE': 'ice-cream bucket',
 'NIFTY2510223600PE': 'ice-cream bucket',
 'NIFTY25MAR24000PE': 'ice-cream bucket',
 'RELIANCE25JAN1310CE': 'covered call',
 'HDFCBANK25JAN1780CE': 'covered call',
 'TITAN25JAN3650CE': 'covered call',
 'INFY25JAN2040CE': 'covered call',
 'TITAN25JAN3500CE': 'covered call',
 'RELIANCE25JAN1260CE': 'covered call',
 'INFY25JAN1940CE': 'covered call',
 'TITAN25JAN3350CE': 'covered call',
 'HDFCBANK25JAN1860CE': 'covered call',
 'LT25JAN3800CE': 'covered call',
 'DLF25JAN830CE': 'iron condor',
 'DLF25J

In [62]:
import pandas as pd
df = pd.DataFrame(output.items(), columns=['symbol', 'strategy'])
df
df.to_csv('strategies.csv', index=False)